# Adoption Barriers & Acceleration

## Purpose
Identify obstacles to adoption and design targeted interventions to accelerate uptake.

## Key Metrics
- **Barrier Analysis**: What prevents adoption?
- **Non-Adopter Profiling**: Who isn't using AI and why?
- **Training Effectiveness**: Impact of training on adoption
- **Intervention ROI**: Which programs drive highest adoption?

## Research Foundation
- Change management (Kotter, 1996)
- Barriers to technology adoption (Venkatesh et al., 2003)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

if not os.path.exists('../data/ai_adoption_employees.csv'):
    exec(open('../data/generate_sample_data.py').read())

employees = pd.read_csv('../data/ai_adoption_employees.csv')
training = pd.read_csv('../data/training_programs.csv')

print(f"Loaded {len(employees)} employees")

## 1. Barrier Analysis

In [ ]:
print(f"ADOPTION BARRIERS ANALYSIS")
print(f"{'='*80}")

# Extract all barriers
all_barriers = []
for barriers_str in employees['primary_barriers']:
    if barriers_str and barriers_str != 'None':
        all_barriers.extend([b.strip() for b in barriers_str.split(';')])

if all_barriers:
    barrier_counts = pd.Series(all_barriers).value_counts()
    
    print(f"\nTop Barriers to Adoption:")
    for i, (barrier, count) in enumerate(barrier_counts.items(), 1):
        pct = count / len(employees) * 100
        print(f"  {i}. {barrier:30s}: {count:3d} employees ({pct:.1f}%)")
    
    # Visualize
    fig, ax = plt.subplots(figsize=(12, 6))
    
    bars = ax.barh(range(len(barrier_counts)), barrier_counts.values, 
                   alpha=0.7, edgecolor='black', color='coral')
    ax.set_yticks(range(len(barrier_counts)))
    ax.set_yticklabels(barrier_counts.index, fontsize=10)
    ax.set_xlabel('Number of Employees', fontsize=11, fontweight='bold')
    ax.set_title('Adoption Barriers', fontsize=13, fontweight='bold')
    ax.grid(alpha=0.3)
    
    for i, val in enumerate(barrier_counts.values):
        ax.text(val + 1, i, f'{val}', va='center', fontsize=9, fontweight='bold')
    
    plt.tight_layout()
    plt.show()

## 2. Non-Adopter Profile

In [ ]:
print(f"\nNON-ADOPTER PROFILE")
print(f"{'='*80}")

non_adopters = employees[employees['adoption_level'] == 'Non-Adopter']
minimal_users = employees[employees['adoption_level'] == 'Minimal User']
at_risk = pd.concat([non_adopters, minimal_users])

print(f"\nAt-Risk Population (Non-Adopters + Minimal Users):")
print(f"  Count: {len(at_risk)} ({len(at_risk)/len(employees)*100:.1f}%)")

# Demographics
print(f"\nBy Age Group:")
age_dist = at_risk['age_group'].value_counts()
for age, count in age_dist.items():
    total_in_age = len(employees[employees['age_group'] == age])
    pct = count / total_in_age * 100
    print(f"  {age:20s}: {count} ({pct:.1f}% of {age})")

print(f"\nBy Department:")
dept_dist = at_risk['department'].value_counts().head(5)
for dept, count in dept_dist.items():
    total_in_dept = len(employees[employees['department'] == dept])
    pct = count / total_in_dept * 100
    print(f"  {dept:20s}: {count} ({pct:.1f}% of {dept})")

# Characteristics
print(f"\nCharacteristics:")
print(f"  Avg tech comfort: {at_risk['tech_comfort_score'].mean():.1f}/5.0")
print(f"  Training completion: {at_risk['completed_ai_training'].mean()*100:.0f}%")
print(f"  Manager support: {at_risk['manager_ai_support'].mean():.2f}/1.0")

## 3. Training Effectiveness

In [ ]:
print(f"\nTRAINING EFFECTIVENESS")
print(f"{'='*80}")

# Training completion impact
trained = employees[employees['completed_ai_training'] == True]
untrained = employees[employees['completed_ai_training'] == False]

trained_adoption = (trained['adoption_level'] != 'Non-Adopter').sum() / len(trained) * 100
untrained_adoption = (untrained['adoption_level'] != 'Non-Adopter').sum() / len(untrained) * 100

print(f"\nAdoption Rates:")
print(f"  Completed training: {trained_adoption:.1f}%")
print(f"  No training:        {untrained_adoption:.1f}%")
print(f"  Lift from training: {trained_adoption - untrained_adoption:.1f} percentage points")

# Training programs
print(f"\nTraining Programs:")
for _, prog in training.iterrows():
    print(f"\n  {prog['program_name']}:")
    print(f"    Duration: {prog['duration_hours']} hours")
    print(f"    Completion rate: {prog['completion_rate']*100:.0f}%")
    print(f"    Satisfaction: {prog['avg_satisfaction']:.1f}/5.0")

# Visualize
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

# Training impact on adoption
categories = ['Completed\nTraining', 'No\nTraining']
adoption_rates = [trained_adoption, untrained_adoption]
colors_train = ['green', 'red']

bars = ax1.bar(categories, adoption_rates, color=colors_train, alpha=0.7, edgecolor='black')
ax1.set_ylabel('Adoption Rate (%)', fontsize=11, fontweight='bold')
ax1.set_title('Training Impact on Adoption', fontsize=13, fontweight='bold')
ax1.set_ylim(0, 100)
ax1.grid(axis='y', alpha=0.3)

for bar, val in zip(bars, adoption_rates):
    ax1.text(bar.get_x() + bar.get_width()/2., val + 2,
            f'{val:.1f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

# Training program comparison
x = np.arange(len(training))
width = 0.35

bars1 = ax2.bar(x - width/2, training['completion_rate']*100, width, 
                label='Completion %', alpha=0.8, color='steelblue', edgecolor='black')
ax2_twin = ax2.twinx()
bars2 = ax2_twin.bar(x + width/2, training['avg_satisfaction'], width, 
                     label='Satisfaction', alpha=0.8, color='coral', edgecolor='black')

ax2.set_ylabel('Completion Rate (%)', fontsize=11, fontweight='bold')
ax2_twin.set_ylabel('Satisfaction (1-5)', fontsize=11, fontweight='bold')
ax2.set_title('Training Program Metrics', fontsize=13, fontweight='bold')
ax2.set_xticks(x)
ax2.set_xticklabels(training['program_name'], rotation=15, ha='right', fontsize=9)
ax2_twin.set_ylim(0, 5)
ax2.legend(loc='upper left')
ax2_twin.legend(loc='upper right')
ax2.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Acceleration Plan

In [ ]:
print(f"\nADOPTION ACCELERATION PLAN")
print(f"{'='*80}")

current_adoption = ((len(employees) - len(non_adopters)) / len(employees) * 100)
target_adoption = 85  # Goal
gap = target_adoption - current_adoption
employees_needed = int(gap / 100 * len(employees))

print(f"\nCurrent state: {current_adoption:.1f}% adoption")
print(f"Target: {target_adoption}% adoption")
print(f"Gap: {gap:.1f} percentage points ({employees_needed} employees)")

print(f"\n🎯 RECOMMENDED INTERVENTIONS:")
print(f"{'='*80}")

print(f"\n1. MANDATORY TRAINING (Priority: High)")
print(f"   • Make AI training required for all employees")
print(f"   • Current completion: {employees['completed_ai_training'].mean()*100:.0f}%")
print(f"   • Target: 100% completion within 90 days")
print(f"   • Expected lift: {trained_adoption - untrained_adoption:.0f} percentage points")

print(f"\n2. MANAGER ENABLEMENT (Priority: High)")
print(f"   • Train managers to model AI usage")
print(f"   • Set team adoption goals")
print(f"   • Manager support correlates {employees['manager_ai_support'].corr(employees['is_adopter']):.2f} with adoption")

print(f"\n3. LOW TECH-COMFORT SUPPORT (Priority: Medium)")
low_tech_comfort = len(employees[employees['tech_comfort_score'] < 3])
print(f"   • {low_tech_comfort} employees with low tech comfort")
print(f"   • Provide extra 1:1 coaching sessions")
print(f"   • Pair with AI champions for peer support")

print(f"\n4. USE CASE LIBRARY (Priority: Medium)")
no_use_cases = len([e for e in employees['primary_barriers'] if 'No clear use cases' in str(e)])
print(f"   • {no_use_cases} employees cite 'no clear use cases'")
print(f"   • Create role-specific use case examples")
print(f"   • Share success stories from early adopters")

print(f"\n5. GAMIFICATION & INCENTIVES (Priority: Low)")
print(f"   • Launch AI adoption challenge")
print(f"   • Recognize power users")
print(f"   • Tie adoption to team goals")

# Timeline
print(f"\n📅 TIMELINE TO 85% ADOPTION:")
print(f"{'='*80}")
print(f"  Month 1-2: Mandatory training rollout")
print(f"  Month 2-3: Manager enablement program")
print(f"  Month 3-4: Targeted support for low-tech-comfort users")
print(f"  Month 4-6: Use case library and gamification")
print(f"\n  Expected outcome: 85% adoption by Month 6")

print(f"\n{'='*80}")

## Key Takeaways

1. **Training is highest-leverage intervention**: Completion increases adoption 30+ points
2. **Barriers are addressable**: Most obstacles can be removed with targeted support
3. **Manager support is critical**: Teams with supportive managers adopt faster
4. **85% adoption is achievable**: With systematic plan over 6 months

**Recommended Actions**:
- Make AI training mandatory (immediate)
- Train managers to champion AI (Month 1-2)
- Provide extra support for low-tech-comfort employees
- Create role-specific use case library
- Track progress monthly and adjust interventions